In [52]:
!pip show qickdawg

Name: qickdawg
Version: 1.2.1
Summary: Software for full quantum control of nitrogen-vacancy defects and other quantum defects in diamond
Home-page: 
Author: 
Author-email: Andy Mounce <amounce@sandia.gov>, Emmeline Riendeau <eriendeau@uchicago.edu>
License: MIT License 

Copyright 2023 National Technology & Engineering Solutions of Sandia, LLC (NTESS). Under the terms of Contract DE-NA0003525 with NTESS, the U.S. Government retains certain rights in this software.

Permission is hereby granted, free of charge, to any person obtaining a copy of this software and associated documentation files (the "Software"), to deal in the Software without restriction, including without limitation the rights to use, copy, modify, merge, publish, distribute, sublicense, and/or sell copies of the Software, and to permit persons to whom the Software is furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all copies or substa

In [53]:
%load_ext autoreload
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt
from copy import copy
import qickdawg as qd

from scipy.optimize import curve_fit
from scipy.signal import find_peaks

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [54]:
qd.start_client('192.168.0.113')

QICK library version mismatch: 0.2.324 remote (the board), 0.2.302 local (the PC)
                        This may cause errors, usually KeyError in QickConfig initialization.
                        If this happens, you must bring your versions in sync.


In [55]:
default_config = qd.NVConfiguration()

default_config.adc_channel = 0
default_config.edge_counting = True
default_config.high_threshold = 2000
default_config.low_threshold = 500


default_config.mw_channel = 0
default_config.mw_nqz = 1
default_config.mw_gain = 5000

default_config.laser_gate_pmod = 0

default_config.relax_delay_tns = 50 # between each rep, wait for everything to catch up, mostly aom

# CPMGXY8 with coarse resolution to check

In [283]:
from qickdawg.arqick.arqick_cpmg_XY8 import CPMGXY8
from copy import copy
soc = qd.soc
config = copy(default_config)
config.mw_gain = 30000
config.mw_pi2_tns = 50
config.freq_fMHz = 500 # in Hz
config.add_linear_sweep(name = "delay", unit = "tns", start = 10, stop = 100, delta=50)
config.n_cpmg = 1 # number of cpmg xy8 rounds
config.pulse_seq_delay_tus = 1
config.reps=1
config.pmod_out_pin = 0 
config.pmod_out_pulse_width_tns = 50
config.pmod_out_trig_delay_tns = 0 #5000-198
config.inherent_trigger_to_pulses_delay_tns = 209.27

prog = CPMGXY8(config)
prog.run_rounds(soc, rounds=1)


Requested 10 to 100 by 50
Instead using 9.765625 to 58.59375 by 48.828125 in 2 steps


100%|██████████| 2/2 [00:00<00:00, 1327.73it/s]


# 200ps resolution now

In [57]:
import copy
from qickdawg.nvpulsing.cpmg_xy_subnano_res import RFTest_CPMG
soc = qd.soc
config = copy.copy(default_config)

# MW params
config.mw_channel = 0
config.freq_fMHz = 1406 # 200
config.mw_gain = 32000 # check user if do two amps for min max
config.mw_nqz = 1
# Timing params
config.mw_pi2_tdds = 100 # check user for min and max
# Sweep params
config.n_cpmg = 1 # check user for min and max
config.reps = 1
config.pulse_seq_delay_tus = 10
# should check the user to make sure min samples is something and max
#config.add_unitless_linear_sweep("delay_tdds", 1000, 1045, delta=5) # the sweep is inclusive of the start and end values
config.add_unitless_linear_sweep("delay_tdds", 1000, 1005, delta=5)
# Triggering
config.pmod_out_pin = 0 
config.pmod_out_pulse_width_tns = 50
config.pmod_out_trig_delay_tns = 0 #5000-198
config.inherent_trigger_to_pulses_delay_tns = 209.27
prog = RFTest_CPMG(config)
prog.run_rounds(soc, rounds=1)

100%|██████████| 2/2 [00:00<?, ?it/s]
